In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import requests

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

def download_uniprot_class(query, label_name, label_id, size=100):
    params = {
        "query": query,
        "format": "tsv",
        "fields": "accession,protein_name,gene_names,organism_name,length,sequence",
        "size": size
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    df = pd.read_csv(pd.io.common.StringIO(response.text), sep="\t")
    df["label_name"] = label_name
    df["label"] = label_id

    return df

In [ ]:
kinase_df = download_uniprot_class(
    query='reviewed:true AND protein kinase',
    label_name='kinase',
    label_id=0,
    size=100
)

kinase_df.head()

In [ ]:
len(kinase_df)

In [ ]:
gpcr_df = download_uniprot_class(
    query='reviewed:true AND G-protein coupled receptor',
    label_name='GPCR',
    label_id=1,
    size=100
)

len(gpcr_df)

In [ ]:
gpcr_df.head()

In [ ]:
ion_df = download_uniprot_class(
    query='reviewed:true AND ion channel',
    label_name='ion_channel',
    label_id=2,
    size=100
)

len(ion_df)

ion_df.head()

In [ ]:
combined_df = pd.concat(
    [kinase_df, gpcr_df, ion_df],
    ignore_index=True
)

combined_df = combined_df[["Sequence", "label", "label_name"]]

combined_df.head()

In [ ]:
print(combined_df.shape)

combined_df["label_name"].value_counts()

In [ ]:
combined_df = combined_df.rename(
    columns={"Sequence": "sequence"}
)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = {
    "train": train_dataset,
    "test": test_dataset
}

dataset

In [ ]:
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["sequence"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./drug_target_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()

eval_results

In [ ]:
predictions_output = trainer.predict(tokenized_test)

logits = predictions_output.predictions
true_labels = predictions_output.label_ids

predicted_labels = np.argmax(logits, axis=-1)

In [ ]:
target_names = ["kinase", "GPCR", "ion_channel"]

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=target_names
)

print(report)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

disp.plot()
plt.title("Confusion Matrix: Drug Target Class Prediction")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    labels=[0, 1, 2]
)

metrics_df = pd.DataFrame({
    "class": target_names,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support
})

metrics_df

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(metrics_df["class"], metrics_df["f1_score"])
plt.ylim(0, 1)
plt.ylabel("F1-score")
plt.title("F1-score per Drug Target Class")
plt.show()

In [ ]:
trainer.save_model("./final_drug_target_model")
tokenizer.save_pretrained("./final_drug_target_model")

In [ ]:
label_map = {
    0: "kinase",
    1: "GPCR",
    2: "ion_channel"
}

def predict_drug_target_class(sequence):
    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    outputs = model(**inputs)

    predicted_label = np.argmax(outputs.logits.detach().cpu().numpy(), axis=-1)[0]

    return label_map[predicted_label]

In [ ]:
test_sequence = "MELRVLLCWASLAAALEETLLNTKLETADLKWVTFPQVDGQWEELSGLDEEQHSVRTYEVCDGPGD"

prediction = predict_drug_target_class(test_sequence)

print("Predicted drug target class:", prediction)

In [ ]:
trainer.save_model("./final_drug_target_model")
tokenizer.save_pretrained("./final_drug_target_model")

In [ ]:
!zip -r final_drug_target_model.zip final_drug_target_model